# Production Water-Body Segmentation with SAM

**Aereo Data Scientist Intern Assignment — Maanvi Bansal**

This notebook is the training and analysis interface for a modular repository. It uses every image in the assigned train/validation/test split, fine-tunes SAM ViT-B, compares prompt types on the complete test split, selects the segmentation threshold only from validation data, and exports production artifacts.

The original notebook was a useful proof of concept, but it evaluated manually selected points on two images. This version adds data validation, leakage-safe splitting, augmentation, training, experiment tracking, calibration and boundary metrics, confidence intervals, inference, Docker/API packaging, tests, and CI.

## Scientific protocol

- **Deployable primary experiment:** no-prompt SAM domain adaptation. No ground-truth information is required at inference.
- **Interactive experiments:** one point, multiple positive/negative points, box, and box-plus-points.
- **Oracle warning:** prompts generated from test masks measure prompt sensitivity; they are not available in automatic deployment.
- **Full-data rule:** no sample-count limiter is used. Every row in each split is processed.
- **Model selection:** validation IoU determines the checkpoint and validation threshold. Test data is used once for final reporting.

In [ ]:
from pathlib import Path
import os
import shutil

PROJECT_DIR = Path("/kaggle/working/aereo-water-sam")

if not PROJECT_DIR.exists():
    candidate_zips = list(Path("/kaggle/input").rglob("aereo-water-sam*.zip"))
    if not candidate_zips:
        raise FileNotFoundError(
            "Upload aereo-water-sam.zip as a Kaggle dataset, or clone the repository "
            "to /kaggle/working/aereo-water-sam."
        )
    print("Extracting:", candidate_zips[0])
    shutil.unpack_archive(str(candidate_zips[0]), "/kaggle/working")

print("Project:", PROJECT_DIR)
print("Files:", len(list(PROJECT_DIR.rglob("*"))))

## Safe Kaggle installation

This installs only packages that Kaggle may not already provide. It intentionally does **not** upgrade NumPy, SciPy, OpenCV, scikit-learn, or Albumentations, avoiding the binary mismatch encountered in the earlier notebook.

In [ ]:
%pip install -q -r /kaggle/working/aereo-water-sam/requirements/kaggle.txt
%pip install -q -e /kaggle/working/aereo-water-sam --no-deps

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

The default configuration uses SAM ViT-B, batch size 2, gradient accumulation 4, AMP, mask-decoder fine-tuning, mixed prompt training, twelve maximum epochs, and early stopping. ViT-B is deliberate: ViT-H is much more likely to run out of memory on a Kaggle T4/P100 and is slower to serve.

In [ ]:
from waterseg.config import load_config

CONFIG_PATH = PROJECT_DIR / "configs" / "sam_vit_b.yaml"
cfg = load_config(CONFIG_PATH)
cfg.to_dict()

### Optional experiment changes

Run the default experiment first. For a second, more expensive domain-adaptation run, change:

```python
cfg.model.trainable_parts = "mask_decoder_and_last_blocks"
cfg.model.unfreeze_last_vision_blocks = 2
cfg.train.encoder_learning_rate = 1e-6
cfg.train.epochs = 6
cfg.tracking.run_name = "sam-vit-b-last-two-blocks"
```

Do not change the split seed between model comparisons.

## 1. Data ingestion, validation, manifest, and split

The preparation command recursively pairs image and mask stems, checks decodability and shape consistency, calculates water coverage and SHA-256 hashes, identifies duplicates, stratifies by water fraction, and prevents duplicate content from crossing splits.

In [ ]:
!python -m waterseg.cli.prepare --config {CONFIG_PATH}

In [ ]:
OUTPUT_DIR = Path(cfg.paths.output_dir)
manifest = pd.read_csv(OUTPUT_DIR / "manifest.csv")
train_df = pd.read_csv(OUTPUT_DIR / "train.csv")
val_df = pd.read_csv(OUTPUT_DIR / "val.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test.csv")

summary = pd.DataFrame({
    "split": ["train", "validation", "test", "total"],
    "images": [len(train_df), len(val_df), len(test_df), len(manifest)],
    "mean_water_fraction": [
        train_df.water_fraction.mean(),
        val_df.water_fraction.mean(),
        test_df.water_fraction.mean(),
        manifest.water_fraction.mean(),
    ],
    "empty_masks": [
        (~train_df.has_water.astype(bool)).sum(),
        (~val_df.has_water.astype(bool)).sum(),
        (~test_df.has_water.astype(bool)).sum(),
        (~manifest.has_water.astype(bool)).sum(),
    ],
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    ax.hist(frame.water_fraction, bins=30, alpha=0.45, label=name)
ax.set_title("Water-pixel coverage by split")
ax.set_xlabel("Water fraction")
ax.set_ylabel("Images")
ax.legend()
plt.show()

In [ ]:
shape_counts = manifest.groupby(["height", "width"]).size().sort_values(ascending=False).rename("count").reset_index()
shape_counts.head(15)

In [ ]:
from waterseg.data.manifest import read_mask, read_rgb

sample_rows = manifest.sample(min(6, len(manifest)), random_state=cfg.train.seed)
fig, axes = plt.subplots(len(sample_rows), 3, figsize=(14, 4 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = np.expand_dims(axes, 0)
for row_axes, (_, row) in zip(axes, sample_rows.iterrows()):
    image = read_rgb(row.image_path)
    mask = read_mask(row.mask_path)
    row_axes[0].imshow(image)
    row_axes[0].set_title(row.image_id)
    row_axes[1].imshow(mask, cmap="gray")
    row_axes[1].set_title(f"Ground truth | water={mask.mean():.3f}")
    row_axes[2].imshow(image)
    row_axes[2].imshow(mask, alpha=0.4, cmap="Blues")
    row_axes[2].set_title("Overlay")
    for axis in row_axes:
        axis.axis("off")
plt.tight_layout()
plt.show()

## 2. Prompt-generation sanity check

Prompts are deterministic per image and prompt mode during evaluation. Training samples prompt modes according to the configured curriculum. Positive points are sampled inside eroded water regions; negative points are sampled safely inside background regions; boxes are jittered to avoid an unrealistically perfect box-only training signal.

In [ ]:
from waterseg.prompting import build_prompt_batch
from waterseg.utils import stable_int_hash

row = train_df.iloc[0]
image = read_rgb(row.image_path)
mask = read_mask(row.mask_path)

modes = ["point1", "points", "box", "box_points"]
fig, axes = plt.subplots(1, len(modes), figsize=(20, 5))
for axis, mode in zip(axes, modes):
    rng = np.random.default_rng(stable_int_hash(f"{row.image_id}:{mode}", cfg.train.seed))
    prompt = build_prompt_batch(
        [mask], mode, cfg.prompts.positive_points, cfg.prompts.negative_points,
        cfg.prompts.box_jitter_fraction, [rng]
    )
    axis.imshow(image)
    axis.imshow(mask, alpha=0.25, cmap="Blues")
    if prompt.points is not None:
        for (x, y), label in zip(prompt.points[0], prompt.labels[0]):
            axis.scatter(x, y, marker="*", s=130, label="positive" if label == 1 else "negative")
    if prompt.boxes is not None:
        x0, y0, x1, y1 = prompt.boxes[0]
        axis.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, linewidth=2))
    axis.set_title(mode)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Full-dataset SAM fine-tuning

This runs on **all training images** and validates on **all validation images** each epoch. The model trains a weighted BCE + Dice + focal objective and an IoU-head regression objective. It uses AMP, gradient accumulation, gradient clipping, cosine decay with warm-up, early stopping, compact checkpoints, and local or W&B tracking.

The primary validation prompt is `none`, because this is the only fully automatic deployment mode in the current project.

In [ ]:
!python -m waterseg.cli.train --config {CONFIG_PATH}

In [ ]:
history = pd.read_csv(OUTPUT_DIR / "training_history.csv")
history.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history.epoch, history.train_loss, marker="o", label="train loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training loss")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history.epoch, history.val_iou, marker="o", label="validation macro IoU at 0.5")
ax.plot(history.epoch, history.val_threshold_iou, marker="o", label="validation IoU after threshold tuning")
ax.set_xlabel("Epoch")
ax.set_ylabel("IoU")
ax.set_title("Validation performance")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 4. Validation threshold selection

A probability threshold of 0.5 is not assumed to be optimal. The best checkpoint stores the threshold selected on validation data. That threshold is then frozen for every final test comparison.

In [ ]:
import json
best_metadata = json.loads((OUTPUT_DIR / "best_model.json").read_text())
selected_threshold = best_metadata["threshold"]
print("Best epoch:", best_metadata["epoch"])
print("Selected validation threshold:", selected_threshold)

threshold_table = pd.read_csv(OUTPUT_DIR / f"threshold_sweep_epoch_{best_metadata['epoch']:02d}.csv")
threshold_table.sort_values("iou", ascending=False).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for metric in ["iou", "dice", "precision", "recall"]:
    ax.plot(threshold_table.threshold, threshold_table[metric], marker="o", label=metric)
ax.axvline(selected_threshold, linestyle="--", label=f"selected={selected_threshold:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metric")
ax.set_title("Validation threshold sweep")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 5. Complete test-set prompt comparison

This evaluates every configured prompt type on every test image and exports aggregate metrics, per-image metrics, HD95/ASSD, latency, and bootstrap confidence intervals.

In [ ]:
!python -m waterseg.cli.evaluate --config {CONFIG_PATH}

In [ ]:
prompt_results = pd.read_csv(OUTPUT_DIR / "evaluation" / "prompt_comparison.csv")
columns = [
    "prompt_mode", "iou", "dice", "precision", "recall", "specificity",
    "balanced_accuracy", "mcc", "cohen_kappa", "mean_boundary_f1",
    "mean_hd95", "ece", "brier_score", "auroc_hist", "auprc_hist",
    "mean_latency_ms"
]
prompt_results[[column for column in columns if column in prompt_results.columns]]

In [ ]:
plot_metrics = ["iou", "dice", "mean_boundary_f1", "mcc"]
comparison = prompt_results.set_index("prompt_mode")[[m for m in plot_metrics if m in prompt_results]].copy()
comparison.plot(kind="bar", figsize=(12, 6))
plt.title("Full test-set prompt comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.show()

### How to interpret the comparison

- `none` is the automatic production result.
- `point1` estimates the value of one human click.
- `points` tests multiple foreground points plus a background correction point.
- `box` estimates coarse region-of-interest guidance.
- `box_points` tests the richest interaction.

Do not present oracle-prompt performance as the automatic model score.

## 6. Failure-case analysis

The next cell inspects the worst no-prompt examples, rather than showing only successful outputs.

In [ ]:
from waterseg.inference import WaterSegmenter

checkpoint_path = OUTPUT_DIR / "checkpoints" / "best.pt"
segmenter = WaterSegmenter(checkpoint_path, device="auto")
no_prompt_rows = pd.read_csv(OUTPUT_DIR / "evaluation" / "test_per_image_none.csv")
worst = no_prompt_rows.nsmallest(min(6, len(no_prompt_rows)), "iou")
test_lookup = test_df.set_index("image_id")

fig, axes = plt.subplots(len(worst), 4, figsize=(18, 4 * len(worst)))
if len(worst) == 1:
    axes = np.expand_dims(axes, 0)
for row_axes, (_, metric_row) in zip(axes, worst.iterrows()):
    item = test_lookup.loc[metric_row.image_id]
    image = read_rgb(item.image_path)
    target = read_mask(item.mask_path)
    prediction, probability = segmenter.segment(image)
    row_axes[0].imshow(image); row_axes[0].set_title(metric_row.image_id)
    row_axes[1].imshow(target, cmap="gray"); row_axes[1].set_title("Ground truth")
    row_axes[2].imshow(prediction, cmap="gray"); row_axes[2].set_title(f"Prediction | IoU={metric_row.iou:.3f}")
    row_axes[3].imshow(probability, vmin=0, vmax=1, cmap="viridis"); row_axes[3].set_title("Water probability")
    for axis in row_axes:
        axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
error_analysis = no_prompt_rows.assign(
    absolute_area_error=lambda frame: (frame.predicted_water_fraction - frame.water_fraction).abs()
)
error_analysis[[
    "image_id", "iou", "dice", "boundary_f1", "hd95",
    "water_fraction", "predicted_water_fraction", "absolute_area_error"
]].sort_values("iou").head(20)

## 7. Production inference smoke test

This verifies the same checkpoint through the reusable inference class and writes a binary mask artifact.

In [ ]:
example = test_df.iloc[0]
image = read_rgb(example.image_path)
mask, probability = segmenter.segment(image)
smoke_output = OUTPUT_DIR / "inference_smoke_mask.png"
cv2.imwrite(str(smoke_output), mask * 255)
print("Saved:", smoke_output)
print("Predicted water fraction:", mask.mean())

## 8. API and container contract

The repository includes:

- `POST /segment` accepting an uploaded image and optional point/box prompts;
- `GET /health` and `GET /ready`;
- JSON logs and latency headers;
- model loading once at service startup;
- cached image embeddings for repeated prompt interactions;
- Docker and Docker Compose files;
- one model replica per worker to avoid accidental GPU-memory duplication.

Build after copying `best.pt` to `artifacts/checkpoints/best.pt`:

```bash
docker build -t aereo-water-sam .
docker run --rm -p 8000:8000 \
  -v "$PWD/artifacts/checkpoints:/models:ro" \
  -e MODEL_CHECKPOINT=/models/best.pt aereo-water-sam
```

## 9. Quality checks

These unit tests cover deterministic leakage-safe splitting, prompt generation, metrics, and tiled stitching without downloading the SAM checkpoint.

In [ ]:
!cd {PROJECT_DIR} && PYTHONPATH=src pytest -q

## 10. Export submission artifacts

The output archive contains manifests, split files, resolved configuration, training history, threshold sweeps, the compact best checkpoint, prompt-comparison tables, confidence intervals, per-image results, and the smoke-test mask.

In [ ]:
archive_base = "/kaggle/working/aereo-water-sam-results"
shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Created:", archive_base + ".zip")

## 11. Optional W&B hyperparameter sweep

After the default run is stable, initialize a W&B sweep from the repository root. Every run reuses the exact same data registry and writes to an isolated `sweeps/<run_id>` directory.

```bash
wandb sweep configs/wandb_sweep.yaml
wandb agent <entity/project/sweep_id>
```

The sweep searches learning rate, weight decay, loss weights, and the proportion of no-prompt batches while optimizing validation IoU after threshold tuning.

## 12. Resume, tiling, and GeoTIFF paths

- To resume an interrupted run, set `train.resume_checkpoint` to `checkpoints/last.pt`. Optimizer, scheduler, AMP scaler, early-stopping state, and RNG states are restored.
- For genuinely large rasters, set `data.materialize_tiles: true`. Parent images are split before tiling, preventing cross-split tile leakage.
- The inference CLI supports overlap-tile stitching. With the optional Rasterio dependency, GeoTIFF output preserves the source CRS and affine transform.


## Next experiments after the core submission is stable

1. Add a semantic baseline such as U-Net or SegFormer under the identical split and metrics.
2. Add Sentinel-2 NIR and SWIR bands and compare RGB against NDWI/MNDWI-aware inputs.
3. Use geographically separated train/test regions instead of a random image split.
4. Run a second-stage fine-tune with the final two SAM vision blocks unfrozen.
5. Add uncertainty-based human prompting: request a click only when confidence or boundary quality is poor.
6. Benchmark ONNX/TensorRT or quantized inference on the actual deployment hardware.
7. Add data and prediction drift monitoring once real production imagery is available.